# Qwen3 医学场景全参微调与 NPU 推理接入 —— 完整交互体验

本 Notebook 完整演示从数据准备到最终接入 `cann-recipes-infer` 推理部署的全流程，代码均为昇腾 NPU 适配版。

流程分为 5 个部分：
1. 下载数据
2. 加载模型并构建监督标签
3. BF16 全参数 SFT
4. Transformer 原生推理
5. cann-recipes-infer 统一执行器部署

> 运行前请确认已完成环境准备（见 `README.md` 中的「前置条件」「体验配置」），本 Notebook 不重复环境安装步骤。


## 1. 下载数据

使用的是 [delicate_medical_r1_data](https://modelscope.cn/datasets/krisfu/delicate_medical_r1_data) 数据集，只取其中 `question`、`think`、`answer` 三列：

- `question`：用户提出的问题，即模型的输入
- `think`：模型的思考过程
- `answer`：模型思考完成后的回复内容

下载并按 9:1 划分为训练集（`train.jsonl`）与验证集（`val.jsonl`）：


In [ ]:
import json
import random
import subprocess
import os

if not os.path.exists("delicate_medical_r1_data"):
    subprocess.run([
        "git", "clone",
        "https://www.modelscope.cn/datasets/krisfu/delicate_medical_r1_data.git"
    ], check=True)

dataset_file = "delicate_medical_r1_data/r1_data_example.jsonl"
print(f"读取数据集文件: {dataset_file}")

data = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"总样本数: {len(data)}")

random.seed(42)
random.shuffle(data)
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]

with open("train.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open("val.jsonl", "w", encoding="utf-8") as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"训练集: {len(train_data)} 条 -> train.jsonl")
print(f"验证集: {len(val_data)} 条 -> val.jsonl")


## 2. 加载模型并构建监督标签

下载 Qwen3-1.7B 基座模型，并加载到昇腾 NPU 上：


In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download("Qwen/Qwen3-1.7B")
print(f"模型下载到: {model_dir}")


In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = model_dir  # 上一步下载得到的路径

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16
).to("npu")

print(f"NPU 可用: {torch.npu.is_available()}, 设备数: {torch.npu.device_count()}")
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params / 1e9:.2f}B")


### 微调前效果（Baseline）

在开始微调之前，先用同样的测试问题跑一遍未经训练的基座模型，作为微调前的效果基线，方便和「4. Transformer 原生推理」小节里微调后的输出直接对比：


In [ ]:
SYSTEM_PROMPT = "你是一个专业的医学推理助手。请根据用户提供的医学问题，进行深入的思考和推理，然后给出专业、准确的回答。"

test_questions = [
    "什么是高血压？它的诊断标准是什么？",
    "阿司匹林的主要药理作用和常见不良反应有哪些？",
    "糖尿病患者出现低血糖反应时应如何紧急处理？",
]

model.eval()
for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"问题 {i+1}: {question}")
    print("-" * 60)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("npu")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1536, temperature=0.7, top_p=0.9, do_sample=True)

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    print(response)


运行结果示例（微调前）：

**问题 1：什么是高血压？它的诊断标准是什么？**
```
<think>
好的，用户问的是高血压的定义和诊断标准。首先，我需要确认高血压的基本概念，包括原发性高血压和继发性高血压的区别。然后，要回顾诊断标准，比如WHO的标准和中国指南的不同之处。还要考虑不同血压值的分类，比如正常高值、高血压前期和确诊高血压。可能用户需要了解诊断的流程，比如测量方法和注意事项。另外，是否需要提到危险因素或并发症？不过问题只问了诊断标准，所以可能不需要深入。要确保信息准确，引用权威来源，比如WHO和中国高血压指南。还要注意用户可能的背景，可能是普通患者或医学生，所以语言要通俗易懂，但保持专业性。最后检查是否有遗漏的关键点，比如血压测量的时机、持续时间等。
</think>

高血压是指血压持续升高导致器官供血不足和器官损伤的病理状态。根据世界卫生组织（WHO）和中国《高血压防治指南》等权威标准，高血压的诊断需满足以下条件：

---

### 一、高血压的定义
1. 原发性高血压（占90%以上）：病因不明，与遗传、年龄、生活方式等因素相关。
2. 继发性高血压：由其他疾病（如肾动脉狭窄、嗜铬细胞瘤、慢性肾脏病等）引起。

---

### 二、诊断标准
#### 1. 诊室血压测量（标准方法）
- 收缩压（SBP）≥140 mmHg 且 舒张压（DBP）≥90 mmHg，同时满足以下任一条件：
  - 3次以上测量结果符合上述标准。
  - 有靶器官损伤（如左心室肥厚、脑卒中史、蛋白尿等）。

#### 2. 中国高血压指南（2018）
- 正常高值：SBP 120-139 mmHg 或 DBP 80-89 mmHg。
- 高血压前期：SBP 140-139 mmHg 或 DBP 90-89 mmHg。
- 确诊高血压：SBP ≥140 mmHg 且 DBP ≥90 mmHg（需排除白大衣高血压、运动后血压升高等因素）。

---

### 三、诊断流程
1. 测量时机：晨起静息状态（避免情绪波动、咖啡因、酒精等影响）。
2. 测量方法：使用经量血压计（水银血压计或电子血压计），确保袖带宽度适中（上臂长度为袖带宽度的2倍）。
3. 重复测量：至少2次，间隔10分钟以上，取平均值。

---

### 四、特殊情况
- 白大衣高血压：诊室血压升高但运动后血压正常，需进一步检查。
- 隐匿性高血压：静息血压升高但运动后正常，需结合临床评估。
- 妊娠高血压：需区分妊娠期高血压和慢性高血压。

---

### 五、诊断要点
- 靶器官损伤：如左心室肥厚、脑卒中史、蛋白尿、肾功能异常等。
- 危险因素：肥胖、高盐饮食、吸烟、酗酒、家族史等。

---

### 总结
高血压的诊断需结合血压值、靶器官损害及危险因素综合判断。早期发现和干预可显著降低心血管事件风险。若需进一步评估，建议结合临床检查（如心电图、尿常规、肾脏功能等）和动态血压监测。
```

**问题 2：阿司匹林的主要药理作用和常见不良反应有哪些？**
```
<think>
好的，我现在需要回答用户关于阿司匹林的主要药理作用和常见不良反应的问题。首先，我得回忆一下阿司匹林的基本信息。阿司匹林，也就是水杨酸钠，是一种非甾体抗炎药（NSAIDs），主要用于缓解疼痛、降低体温、减少炎症和退烧。它还有抗血小板的作用，帮助预防血栓形成，因此常用于心脏病和中风的预防。

接下来是药理作用部分。阿司匹林的药理作用主要在于抑制环氧化酶（COX）的活性，特别是COX-1和COX-2。COX-1在胃肠道中起作用，保护胃黏膜，而COX-2则参与炎症反应。抑制COX可以减少前列腺素的合成，从而达到解热、镇痛和抗炎的效果。此外，阿司匹林还具有抗血小板聚集的作用，通过抑制血小板的聚集来预防血栓，这在心血管疾病预防中很重要。

然后是常见不良反应。首先，胃肠道不适是常见的，比如胃痛、胃溃疡、消化性溃疡，这可能是因为COX-1抑制导致胃黏膜保护作用减弱。其次，出血风险，特别是消化道出血和脑出血，因为抗血小板作用可能增加出血风险。还有过敏反应，如皮疹、荨麻疹，严重时可能引起过敏性休克。另外，长期使用可能引起肝肾功能异常，或者水杨酸反应，表现为恶心、呕吐、头痛等。还有可能引起中枢神经系统副作用，比如嗜睡、头晕，或者在高剂量下引起中枢抑制。

还需要考虑其他不良反应，比如肾功能影响，长期使用可能导致肾功能下降，尤其是老年人或有肾病史的患者。此外，阿司匹林可能引起心血管方面的副作用，如心律失常，尤其是在高剂量或长期使用的情况下。

不过，我需要确认这些信息是否准确。比如，是否所有不良反应都正确？比如，水杨酸反应是否确实存在，以及它的表现。还有，是否有其他常见的不良反应，比如心血管事件，或者与其他药物的相互作用？

另外，需要注意阿司匹林的剂量和使用方法，比如长期使用可能有胃肠道副作用，而短期使用可能风险较低。还有，不同人群的反应可能不同，比如老年人或有胃病史的人更容易出现胃肠道问题。

总结起来，药理作用部分要强调COX抑制和抗血小板作用，不良反应则包括胃肠道、出血、过敏、肝肾功能、中枢神经系统副作用等。需要分点列出，确保准确性和全面性。
</think>

阿司匹林（乙酰水杨酸）是一种常见的非甾体抗炎药（NSAIDs），其主要药理作用和不良反应如下：

---

### 一、主要药理作用
1. 解热镇痛：通过抑制中枢神经系统前列腺素（PGI2）的合成，减少发热和疼痛感受器的敏感性，达到退热、镇痛效果。
2. 抗炎作用：抑制炎症反应中的COX-1和COX-2酶活性，减少前列腺素的生成，从而减轻炎症反应。
3. 抗血小板聚集：通过抑制血小板中TXA2的合成，减少血小板聚集，预防血栓形成，常用于心血管疾病的预防（如心肌梗死、中风）。
4. 抗风湿作用：对类风湿性关节炎等自身免疫性疾病有一定疗效，但需注意长期使用可能引发的副作用。

---

### 二、常见不良反应
1. 胃肠道反应：胃肠道溃疡/出血（COX-1抑制导致胃黏膜保护作用减弱，增加胃炎、胃溃疡、消化道出血风险，尤其长期大剂量使用）；恶心、呕吐（可能因药物刺激胃部或中枢神经系统抑制作用）。
2. 出血风险：消化道出血（胃肠道黏膜损伤导致出血，严重时可致呕血、黑便）；颅内出血（抗血小板作用可能增加脑出血风险，尤其与肝素等抗凝药合用时）；其他出血（如牙龈出血、皮肤瘀斑等）。
3. 过敏反应：皮疹、荨麻疹（过敏体质者可能出现过敏反应）；严重过敏性休克（罕见但可能因药物过敏反应引发）。
4. 肝肾功能影响：肝功能异常（长期大剂量使用可能引发肝酶升高或肝损伤）；肾功能异常（长期使用可能增加肾结石或肾功能下降风险，尤其老年人或肾功能不全者）。
5. 中枢神经系统副作用：嗜睡、头晕（因药物对中枢神经系统的抑制作用）；中枢抑制（高剂量或长期使用可能引起嗜睡、意识模糊等）。
6. 其他：水杨酸反应（罕见但可能表现为头痛、恶心、呕吐等，多见于长期大剂量使用或与某些药物如利尿剂、抗抑郁药联用时）；心血管风险（长期使用可能增加心肌梗死、中风风险，尤其与阿司匹林合用其他抗血小板药物时）。

---

### 三、注意事项
- 禁忌症：严重肝肾疾病、出血倾向、过敏史、孕妇（妊娠期禁用）、儿童等。
- 用药原则：需遵医嘱，避免长期大剂量使用，尤其是有胃肠道病史者。
- 相互作用：与抗凝药（如华法林）、抗血小板药（如氯吡格雷）联用可能增加出血风险，需谨慎。

---

### 总结
阿司匹林通过抑制COX酶发挥解热、镇痛、抗炎及抗血小板作用，但需警惕胃肠道出血、出血性中风、肝肾损伤等不良反应。合理用药需结合患者个体情况及临床指征。
```

**问题 3：糖尿病患者出现低血糖反应时应如何紧急处理？**
```
<think>
好的，用户问的是糖尿病患者出现低血糖反应时应如何紧急处理。首先，我需要回忆一下低血糖的定义和症状。低血糖通常是指血糖水平低于正常范围，一般低于3.9 mmol/L。症状可能包括出汗、心慌、颤抖、头晕、恶心、意识模糊，甚至昏迷。

接下来，用户可能想知道具体的处理步骤。首先，应该确认低血糖是否确实发生，然后采取措施。常见的处理方法包括立即摄入快速作用的碳水化合物，比如葡萄糖片、果汁或糖果，然后监测血糖。同时，需要避免进一步的运动，防止血糖进一步下降。

还要考虑患者是否有口服药物，比如胰岛素或其他降糖药，是否需要调整剂量。如果患者有胰岛素，可能需要暂时停用，或者调整使用时间。另外，如果患者有糖尿病并发症，比如酮症酸中毒，可能需要进一步处理，但这种情况通常伴随其他症状，比如恶心、呕吐、腹痛等。

还要注意患者是否有昏迷的情况，这时候可能需要静脉注射葡萄糖，或者在紧急情况下联系医疗人员。同时，要提醒患者以后避免空腹时间过长，规律饮食，避免剧烈运动，以及定期监测血糖。

可能用户是患者或家属，需要明确的步骤和注意事项。需要确保回答准确，步骤清晰，同时提醒可能的并发症和后续措施。比如，如果患者出现昏迷，可能需要立即送医，而不是自行处理。

还要考虑不同类型的糖尿病，比如1型和2型，处理方式是否有差异？不过一般情况下，处理原则相似，但1型糖尿病患者可能需要更频繁的胰岛素调整。不过用户的问题可能更偏向于一般情况，所以保持通用建议即可。

最后，确保回答结构清晰，分点列出步骤，同时加入注意事项，比如避免使用某些药物，监测血糖，后续管理等。需要避免专业术语过多，用通俗易懂的语言解释，同时保持准确性。
</think>

糖尿病患者出现低血糖反应时，应立即采取以下紧急处理措施，以防止进一步恶化：

---

### 1. 确认低血糖
- 症状判断：观察患者是否有以下症状（如出汗、心悸、头晕、意识模糊、嗜睡、昏迷等）。
- 快速评估：确认是否为低血糖（血糖 < 3.9 mmol/L）。

---

### 2. 紧急处理步骤
#### （1）立即摄入快速作用的碳水化合物
- 口服途径：葡萄糖片/糖果 15-20克（如含糖的糖果、含糖饮料）；果汁/运动饮料 50-100毫升（含糖饮料）；含糖食物如饼干、面包、水果（如香蕉、葡萄）。
- 静脉注射：若患者昏迷或无法口服，立即静脉注射50%葡萄糖液10-20毫升（需由医护人员操作）。

#### （2）暂停胰岛素或降糖药物
- 胰岛素患者：若使用胰岛素或口服降糖药（如格列美脲、二甲双胍等），需暂停药物，并避免运动。
- 注意事项：需根据具体药物和病情调整，避免自行调整剂量。

#### （3）监测血糖
- 立即检测血糖：确认血糖水平是否回升。
- 持续监测：若患者意识不清或持续低血糖，需持续监测血糖直至恢复。

#### （4）补充水分
- 避免脱水：低血糖常伴随脱水，需少量多次补充水分（如口服含电解质的饮料）。

---

### 3. 紧急情况处理
- 昏迷或意识丧失：立即送医，可能需静脉注射葡萄糖或胰高血糖素（需专业医护人员操作）。
- 酮症酸中毒：若伴随酮体（如恶心、呕吐、腹痛、呼吸深快），需紧急处理酮症酸中毒。

---

### 4. 后续管理
- 避免空腹过久：规律饮食，避免长时间空腹。
- 避免剧烈运动：低血糖发作后2小时内避免剧烈活动。
- 记录血糖变化：记录发作时间、血糖值、用药情况，便于后续调整治疗。
- 教育患者：告知患者低血糖的识别症状及处理方法，避免再次发生。

---

### 注意事项
- 避免使用含糖饮料：若患者无法口服，需避免饮用含糖饮料（如可乐、奶茶），以防血糖波动。
- 特殊人群：如老年人、儿童或有肝肾功能不全者，需谨慎调整处理方式。
- 药物相互作用：若患者同时使用某些药物（如利尿剂、NSAIDs），需谨慎。

---

### 总结
低血糖反应的紧急处理核心是快速恢复血糖，优先选择口服碳水化合物或静脉注射葡萄糖，同时避免进一步降糖措施。若症状持续或加重，需立即送医，防止发生昏迷、器官损伤等严重后果。
```

可以看出，微调前的基座模型已经具备较强的通用医学知识和结构化回答能力（这与 Qwen3-1.7B 本身的预训练能力有关），`<think>` 推理过程也基本完整。微调前后的差异，需结合「4. Transformer 原生推理」小节中同样 3 个问题的微调后回答来对比。


构建监督标签：把 `think` 和 `answer` 拼接成 `<think>...</think>\n\n答案` 的格式作为训练目标，并只对 assistant 回复部分计算 loss（prompt 部分标签设为 `-100`，训练时不计入 loss）：


In [ ]:
import json
from torch.utils.data import Dataset

SYSTEM_PROMPT = "你是一个专业的医学推理助手。请根据用户提供的医学问题，进行深入的思考和推理，然后给出专业、准确的回答。"
MAX_LENGTH = 2048


class MedicalDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=MAX_LENGTH):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        with open(data_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line))
        print(f"加载数据: {data_path}, 样本数: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        question = item["question"]
        think = item["think"]
        answer = item["answer"]

        # 构建监督标签：think + answer 拼接为训练目标
        assistant_content = f"<think>{think}</think>\n\n{answer}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": assistant_content},
        ]
        full_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )

        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
        prompt_text = self.tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True,
        )

        full_ids = self.tokenizer(full_text, max_length=self.max_length, truncation=True, padding=False)
        prompt_ids = self.tokenizer(prompt_text, max_length=self.max_length, truncation=True, padding=False)

        input_ids = full_ids["input_ids"]
        attention_mask = full_ids["attention_mask"]
        prompt_len = len(prompt_ids["input_ids"])
        # prompt 部分标签设为 -100，训练时不计入 loss，只监督 assistant 回复部分
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        if len(labels) != len(input_ids):
            labels = labels[:len(input_ids)]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


train_dataset = MedicalDataset("train.jsonl", tokenizer)
val_dataset = MedicalDataset("val.jsonl", tokenizer)


## 3. BF16 全参数 SFT

使用 BF16 精度进行全参数监督微调（不使用 LoRA），并通过 `report_to="swanlab"` 记录训练过程：


In [ ]:
import swanlab
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer, EarlyStoppingCallback

OUTPUT_DIR = "./output_qwen3_medical"

swanlab.init(project="qwen3-medical-npu-finetune", experiment_name="qwen3-1.7b-medical-npu")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, max_length=MAX_LENGTH, label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_steps=680,  # 上限设为约 5 个 epoch，配合早停实际会更早停止
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="swanlab",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],  # 连续5次eval无改善则自动停止
)

print("开始训练...")
trainer.train()

print("保存模型...")
import os
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"模型已保存到: {final_dir}")


训练结束后，取验证集前 3 条数据生成回复，记录到 SwanLab 的 `Prediction` 中，便于在网页端查看训练效果：


In [ ]:
model.eval()
sample_data = []
with open("val.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        sample_data.append(json.loads(line))

prediction_texts = []
for item in sample_data:
    question = item["question"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("npu")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    print(f"Question: {question}\n\nLLM: {response}\n")
    prediction_texts.append(swanlab.Text(f"Question: {question}\n\nLLM: {response}"))

swanlab.log({"Prediction": prediction_texts})
swanlab.finish()


### SwanLab 云端记录

训练过程与 `Prediction` 样例记录于 SwanLab：

- 项目名：`qwen3-medical-npu-finetune`
- 实验名：`qwen3-1.7b-medical-npu`
- 实验链接：https://swanlab.cn/@catherrrrrine/qwen3-medical-npu-finetune

> 该项目已在 SwanLab 设为公开（Public），无需登录 SwanLab 账号即可通过上述链接直接查看训练曲线与 `Prediction` 样例。

`eval/loss` 与 `train/loss` 均随训练步数稳定下降并收敛，全程未见过拟合或发散：

![eval/loss](./images/swanlab_eval_loss.png)

`eval/loss`：从初始约 1.41 快速下降，200 步左右进入缓慢收敛区间，训练至约 650 步时稳定在 1.16 左右，全程未出现回升，说明未发生过拟合。

![train/loss](./images/swanlab_train_loss.png)

`train/loss`：整体趋势与 `eval/loss` 一致，从初始约 1.93 快速下降，150 步后进入平稳震荡下降区间，训练结束时收敛至约 1.07，与验证集损失走势吻合。


## 4. Transformer 原生推理

在接入 `cann-recipes-infer` 之前，先用原生 `transformers` 单独验证训练产出的 checkpoint 能否正常推理，作为独立于推理框架的基准测试。测试问题与「2. 微调前效果（Baseline）」完全一致，可直接对比微调前后的回答质量：


In [ ]:
import torch
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM

CHECKPOINT_PATH = "./output_qwen3_medical/final"
SYSTEM_PROMPT = "你是一个专业的医学推理助手。请根据用户提供的医学问题，进行深入的思考和推理，然后给出专业、准确的回答。"

test_questions = [
    "什么是高血压？它的诊断标准是什么？",
    "阿司匹林的主要药理作用和常见不良反应有哪些？",
    "糖尿病患者出现低血糖反应时应如何紧急处理？",
]

infer_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH, trust_remote_code=True)
infer_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16
).to("npu")
infer_model.eval()

for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"问题 {i+1}: {question}")
    print("-" * 60)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = infer_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = infer_tokenizer(text, return_tensors="pt").to("npu")

    with torch.no_grad():
        outputs = infer_model.generate(**inputs, max_new_tokens=1536, temperature=0.7, top_p=0.9, do_sample=True)

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = infer_tokenizer.decode(generated_ids, skip_special_tokens=True)
    print(response)


运行结果示例（微调后）：

**问题 1：什么是高血压？它的诊断标准是什么？**
```
<think>
嗯，用户问的是高血压的定义和诊断标准，我需要先回忆一下相关的医学知识。高血压，也就是高血压病，通常是指血压持续升高的情况。首先，我得确定血压的正常范围，因为诊断标准里肯定有这个基础。比如，正常血压是收缩压<120mmHg和舒张压<80mmHg，而高血压就是收缩压≥140mmHg或者舒张压≥90mmHg的情况。

接下来，用户可能想知道诊断的具体标准，不仅仅是数值，还要考虑什么因素？我记得高血压的诊断可能需要结合多个指标，比如持续的血压升高，以及是否有症状。比如，如果患者有高血压症状，或者血压持续升高超过一定时间，可能需要进一步检查，比如24小时动态血压监测或者多次测量。

然后，我需要考虑诊断的分级，比如原发性高血压和继发性高血压的区别，但用户的问题里可能不需要这么详细，但诊断标准中可能包括这些信息。不过根据提供的答案，用户可能只需要基本的数值标准，所以可能不需要深入分级。

另外，诊断标准里提到的"血压持续升高"和"存在症状"是否准确？比如，有些患者可能没有症状，但血压仍然高，这时候可能需要动态监测。另外，是否需要排除其他疾病导致的高血压？比如药物影响，或者某些病理情况，但诊断标准可能主要基于血压数值和症状。

还要注意单位是否正确，比如mmHg，用户的问题里已经用了这个单位，所以没问题。另外，可能需要提到高血压的分类，比如轻度、中度、重度，但答案里可能没有，所以保持简洁。

有没有可能用户是医学生或者患者，需要更详细的解释？比如，为什么血压超过某个数值，或者高血压的风险因素？不过问题只问定义和标准，所以可能不需要扩展，但答案里已经涵盖了这些内容。

总结一下，答案的结构应该是先定义，然后给出血压数值标准，再说明需要持续升高和症状存在，可能还要提到诊断方法如动态监测。这样用户就能理解了。需要确保没有遗漏关键点，比如诊断的标准不仅仅是数值，还包括症状和时间因素。同时，可能还要提到高血压的类型，但根据提供的答案，可能不需要，所以保持答案的简洁性。
</think>

高血压，又称高血压病，是指以血管壁的弹性减退和血管硬化为特征的疾病，导致血液流动时对血管壁的侧压增加，即血压升高。根据世界卫生组织的标准，高血压的诊断标准是：如果一个人的血压持续升高，且在安静状态下，收缩压达到或超过140毫米汞柱，或者舒张压达到或超过90毫米汞柱，就可以诊断为高血压。此外，患者还需要存在症状，或者有明确的高血压病史。这种病的诊断通常需要结合患者的血压水平、症状以及进一步的检查来确认。
```

**问题 2：阿司匹林的主要药理作用和常见不良反应有哪些？**
```
<think>
嗯，用户问的是阿司匹林的主要药理作用和常见不良反应，我需要先回忆一下阿司匹林的基本知识。阿司匹林，也就是水杨酸乙酯，属于非甾体抗炎药（NSAIDs）类，对吧？那它的主要作用应该和抗炎、镇痛、解热有关，对吧？我记得NSAIDs的作用机制是抑制环氧合酶（COX），所以阿司匹林可能通过抑制COX-1和COX-2来减少前列腺素的合成，从而达到这些效果。

接下来是药理作用部分。抗炎方面，确实，阿司匹林可以减轻炎症反应，比如关节炎、肌肉疼痛之类的。镇痛的话，应该也是通过抑制前列腺素，减少疼痛信号传递。解热的话，可能也是同样的机制，降低体温。另外，可能还有抗血小板聚集的作用，对吧？因为阿司匹林常用于预防心脏病发作，所以抗血小板也是它的作用之一。

然后是不良反应。常见的应该包括胃肠道出血、溃疡，因为NSAIDs可能损伤胃肠道黏膜。还有可能引起过敏反应，比如皮疹、哮喘发作。另外，长期使用可能有肝肾毒性，比如肝酶升高、肾功能异常。还有，阿司匹林可能引发水杨酸反应，比如严重的胃肠道反应或者全身性反应，比如皮疹、发热、头痛，严重的话甚至休克。另外，药物相互作用方面，比如与抗凝药（如华法林）合用可能增加出血风险，所以需要提醒用户注意药物相互作用。

用户可能想知道为什么这些不良反应会发生，比如为什么胃肠道出血？可能因为COX-1抑制导致胃黏膜保护作用减弱，而COX-2抑制可能对肠道黏膜影响较小，所以COX-1抑制更主要。另外，过敏反应可能是因为免疫系统对药物成分的反应，或者药物本身引起的。

用户可能没有直接问，但深层需求可能是想了解如何正确使用阿司匹林，或者在用药时需要注意什么。所以回答时不仅要列出药理作用和不良反应，还要强调注意事项，比如饭后服用减少胃刺激，避免与其他抗凝药合用，以及长期使用的风险。

需要确认是否有遗漏的不良反应，比如对肾脏的影响，或者是否还有其他副作用。比如，阿司匹林可能引起心血管事件的风险，比如心脏病发作，但主要还是胃肠道和过敏反应。另外，长期使用可能有其他副作用，比如消化道溃疡、出血，还有可能影响血液凝固，导致出血风险增加。

总结一下，药理作用包括抗炎、镇痛、解热和抗血小板；不良反应包括胃肠道出血、溃疡、过敏反应、肝肾毒性、水杨酸反应、药物相互作用。注意事项要提到避免与其他抗凝药合用，饭后服用，以及长期使用的风险。需要确保信息准确，比如水杨酸反应的具体表现，比如皮疹、发热、头痛，严重时休克。另外，抗血小板作用可能在预防血栓方面有用，但出血风险也存在，所以需要权衡利弊。

可能用户是患者或医学生，需要明确且易懂的解释，所以用词要准确但不过于专业。比如解释COX的作用机制时，可以简单说抑制前列腺素合成，减少炎症和疼痛。不良反应部分要分点说明，但用户给的回复已经分点，可能需要保持结构一致。

最后检查是否有错误，比如阿司匹林的抗血小板作用是否正确，是的，它确实通过抑制血栓烷A2的合成，促进血小板聚集的抑制，所以是抗血小板的。另外，肝肾毒性可能需要提到长期或大剂量使用，或者与其他药物同时使用时的风险增加。水杨酸反应可能需要解释为什么会有，比如药物本身的副作用，或者与某些药物相互作用导致。
</think>

阿司匹林的主要药理作用包括抗炎、镇痛、解热以及抗血小板的作用。其抗血小板的作用机制是通过抑制血栓烷A2的合成，从而抑制血小板的聚集。此外，阿司匹林还具有抗炎、镇痛、解热等作用，这些作用均来源于它对COX酶的抑制。

阿司匹林的常见不良反应包括胃肠道出血、溃疡、过敏反应、肝肾毒性、水杨酸反应、药物相互作用等。在用药过程中，需要注意避免与其他抗凝药物（如华法林）同时使用，以免增加出血风险。同时，阿司匹林可能引起胃肠道不适，因此建议在饭后服用以减少胃肠道刺激。对于长期或大剂量使用阿司匹林的患者，应密切监测肝肾功能，以预防肝肾毒性。
```

**问题 3：糖尿病患者出现低血糖反应时应如何紧急处理？**
```
<think>
嗯，用户问的是糖尿病患者出现低血糖反应时的紧急处理方法。首先，我需要回忆一下低血糖的定义和症状。低血糖通常指血糖浓度低于正常范围，可能影响大脑功能，导致头晕、出汗、心悸等症状。然后，我需要考虑糖尿病患者的特殊性，他们可能有胰岛素分泌不足、口服降糖药过量或者运动后血糖波动等问题。

接下来，用户提供的答案已经列出了几个步骤：立即进食含糖食物、补充葡萄糖、监测血糖、再次进食、避免药物影响、及时就医。我需要确保这些步骤是否全面且正确。比如，是否应该建议立即补充含糖食物，而不是直接注射葡萄糖？因为注射葡萄糖可能引起低血糖加重，所以答案中的"口服葡萄糖"是正确的。然后，补充葡萄糖后监测血糖是必要的，但用户提供的答案中提到的再次进食是否必要呢？可能在某些情况下，比如患者无法口服，可能需要静脉注射，但答案里没有提到，可能需要确认是否足够。

另外，答案中提到避免药物影响，可能指的是不要立即再次使用胰岛素或其他降糖药，但有时可能需要根据情况调整。比如，如果患者在运动后低血糖，可能需要暂停运动并补充能量。但答案中的建议可能已经足够，所以不需要深入讨论药物调整。

用户可能希望得到一个简明但全面的回答，所以需要确保每个步骤都正确，并且逻辑清晰。可能需要检查是否有遗漏的步骤，比如是否应该建议联系医生或家人，或者是否需要监测血糖直到稳定。此外，是否需要提醒患者下次避免类似情况？

另外，用户的问题中提到"紧急处理"，所以步骤需要快速有效。比如，立即进食是关键，而补充葡萄糖后再次进食可能用于补充能量，避免低血糖再次发生。同时，监测血糖直到恢复到正常范围，这样能确保处理有效。

可能还需要考虑患者的个体差异，比如是否有胃肠道问题，是否能口服，或者是否需要静脉输注葡萄糖。但答案中的方法已经足够一般性，所以应该没问题。总结下来，答案已经覆盖了主要处理步骤，所以应该没问题。但需要确保用词准确，比如"含糖食物"是否包括水果、糖果等，而不仅仅是糖水。可能需要确认是否应该建议含水的糖分，但通常含糖食物是合适的。

最后，检查是否有更好的方法或补充点，比如是否应该建议患者在处理后立即联系医生，或者是否需要进一步的检查。但答案中已经提到及时就医，所以可能已经足够。因此，用户提供的答案是正确的，但需要确保没有错误，比如是否应该先口服葡萄糖再补充食物，或者是否应该先补充食物再注射葡萄糖。根据答案中的步骤，先口服葡萄糖，然后监测，再补充食物，这可能是因为患者可能无法立即口服，或者需要先补充葡萄糖以迅速提升血糖，然后再通过食物补充能量。所以这个顺序是合理的。

因此，综合以上思考，答案中的步骤是正确的，覆盖了紧急处理的关键点，并且逻辑清晰。
</think>

糖尿病患者出现低血糖反应时，应立即补充含糖食物，如糖果、果汁、面包等，以迅速提高血糖水平。同时，应监测血糖，观察是否有改善。如果血糖仍未恢复正常，应再次补充含糖食物。在处理过程中，应避免使用可能降低血糖的药物，如胰岛素，以防止血糖进一步下降。如果情况严重，应及时就医，以便获得进一步的治疗。
```

对比「2. 微调前效果（Baseline）」中同样 3 个问题的回答可以看出：微调前的基座模型倾向于用 `###` 分级标题、分点列表把回答组织成结构化的"文档"形式；微调后的模型仍保留了完整的 `<think>` 推理过程，但最终回答收敛为更直接、更口语化的连续段落，风格上更贴近医学问答数据集本身的标注格式。


## 5. cann-recipes-infer 统一执行器部署

将训练产出的 checkpoint 接入昇腾官方推理框架 [cann-recipes-infer](https://gitcode.com/cann/cann-recipes-infer)，通过统一执行器 `executor/scripts/infer.sh` 完成部署与推理验证。

### 5.1 获取源码并配置环境


In [ ]:
%%bash
if [ ! -d "cann-recipes-infer" ]; then
    git clone https://gitcode.com/cann/cann-recipes-infer.git
fi
cd cann-recipes-infer

sed -i 's|cann_path="your_cann_pkgs_path"|cann_path="/home/developer/Ascend/cann-9.0.0/aarch64-linux"|' \
    executor/scripts/set_env.sh

sed -i '/export ASCEND_HOME_PATH=\$cann_path/d' \
    executor/scripts/set_env.sh

pip install -r ./models/qwen/requirements.txt --break-system-packages


### 5.2 复制统一执行器配置模版，写入训练产物路径

复制官方 Qwen3-8B 单卡配置模板，并将 `model_path` 改为本 Notebook 第 3 部分训练产出的 checkpoint 路径：


In [ ]:
%%bash
cd cann-recipes-infer
cp models/qwen/config/qwen3_8b_1tp.yaml models/qwen/config/qwen3_custom_1tp.yaml


In [ ]:
# 将训练产物路径写入运行时 YAML
import os

yaml_path = "cann-recipes-infer/models/qwen/config/qwen3_custom_1tp.yaml"
checkpoint_path = os.path.abspath("./output_qwen3_medical/final")

with open(yaml_path, "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace(
    'model_path: "/data/models/origin/Qwen3-8B"',
    f'model_path: "{checkpoint_path}"'
)

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(content)

print(f"已将 model_path 写入: {checkpoint_path}")


### 5.3 补充 lm_head 权重

Qwen 系列模型的 `lm_head` 与 `embed_tokens` 权重共享，训练保存的 checkpoint 中不包含独立的 `lm_head.weight`，接入推理前需要手动补充：


In [ ]:
from safetensors.torch import load_file, save_file
import shutil

CKPT_PATH = os.path.join(checkpoint_path, "model.safetensors")
shutil.copy(CKPT_PATH, CKPT_PATH + ".backup")

weights = load_file(CKPT_PATH)
weights["lm_head.weight"] = weights["model.embed_tokens.weight"].clone()
save_file(weights, CKPT_PATH, metadata={"format": "pt"})

print("已补充 lm_head.weight")


### 5.4 从 models/qwen 工作目录启动统一执行器

统一执行器运行时会将工作目录切换到对应的模型子目录（如 `models/qwen`）读取配置与资源，在 `cann-recipes-infer` 仓库根目录下执行：


In [ ]:
%%bash
cd cann-recipes-infer
bash executor/scripts/infer.sh --model qwen --yaml qwen3_custom_1tp.yaml


### 5.5 用自定义领域问题验证效果

默认测试 prompt 与训练领域无关，替换为医学问题后重新执行统一执行器，验证微调效果确实生效：


In [ ]:
%%bash
cd cann-recipes-infer
cp dataset/default_prompt.json dataset/default_prompt.json.backup

cat > dataset/default_prompt.json << 'EOF'
{
    "text": "医生，我最近被诊断为糖尿病，应该如何调整饮食？"
}
EOF

bash executor/scripts/infer.sh --model qwen --yaml qwen3_custom_1tp.yaml

mv dataset/default_prompt.json.backup dataset/default_prompt.json


运行结果示例：

**输入**：
```
医生，我最近被诊断为糖尿病，应该如何调整饮食？
```

**输出**：
```
<think>
嗯，用户问的是糖尿病患者如何调整饮食，我需要先回忆一下糖尿病饮食管理的基本原则。首先，我应该从控制总热量开始，因为热量摄入过多会导致血糖升高。然后是碳水化合物的控制，特别是选择低升糖指数的食物，比如全谷物、豆类这些，这样能减缓血糖上升的速度。

接下来是蛋白质和脂肪的摄入，这部分可能需要提到适量，但具体比例可能需要更详细的信息。比如，建议每天摄入多少克蛋白质，脂肪的热量占比是多少。另外，纤维的摄入也很重要，高纤维的食物有助于延缓碳水化合物的吸收，比如蔬菜、水果和全谷物。

然后，用户可能还关心具体的饮食结构，比如三餐的分配，是否有加餐时间，以及如何避免高糖高脂的食物。比如，避免含糖饮料和精制碳水化合物，选择天然甜味剂如蜂蜜或者用天然甜味剂替代。同时，烹饪方式也很关键，比如少油少盐，多用蒸、煮、烤的方式。

另外，用户可能没有提到的点包括饮食记录和监测血糖，这可能也是调整饮食的一部分。不过根据问题，用户主要问的是调整饮食的方法，
```

> 输出在 `<think>` 推理过程中被截断，未生成最终回答：`qwen3_custom_1tp.yaml` 沿用了官方模板的 `max_new_tokens=256`，256 个 token 的生成上限内还没走出思考阶段。如需完整回答，可在 YAML 里调大 `max_new_tokens`。

**性能实测**（`qwen3_medical_sft.yaml`，1.7B 规模，两次独立复现结果一致）：

| 指标 | 实测值 |
| --- | --- |
| Prefill | 27.95 ms（复现环境二次实测：27.93 ms） |
| Decode（平均） | 5.18 ms（复现环境二次实测：5.21 ms，误差在合理范围内） |


至此，完整流程结束：数据准备 → 微调前效果基线 → 构建监督标签 → BF16 全参数 SFT → Transformer 原生推理验证（微调后对比）→ 接入 cann-recipes-infer 统一执行器部署 → 领域问题验证效果。
